This notebook is for the process of generating and collecting model responses for professional advice-seeking prompts. Prompts of each professional domain are processed seperately for the sake of available resources.

# Llama-3.1-8B-Instruct Responses

In [ ]:
!pip install --quiet huggingface_hub transformers datasets accelerate torch bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 44.2 MB/s eta 0:00:00


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from huggingface_hub import login

In [ ]:
login()

In [ ]:
# the open-source model evaluated in the thesis
model_name = "meta-llama/Llama-3.1-8B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

# 4-bit quantization for the sake of available resources
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

## Medical Prompt Responses

In [ ]:
from datasets import load_dataset
from transformers import pipeline

In [ ]:
# get the domain specific data
ds = load_dataset("imhann/medical_prompts", split="train")
ds

README.md:   0%|          | 0.00/294 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/89.9k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/300 [00:00<?, ? examples/s]

Dataset({
    features: ['input'],
    num_rows: 300
})

In [ ]:
# function for getting a model response to a prompt
def generate_response(example):
  inputs = tokenizer.apply_chat_template(
        [
            {"role": "user", "content": example["input"]},
        ],
        return_tensors="pt",
        add_generation_prompt=True,
    ).to(model.device)

  with torch.no_grad():
      outputs = model.generate(
          **inputs,
          max_new_tokens=300,
          # to skip warning
          pad_token_id=tokenizer.eos_token_id,
          do_sample=True,
          temperature=0.7,
          top_p=1.0,
      )

  input_length = inputs["input_ids"].shape[-1]

  # decode only the generated part
  response = tokenizer.decode(
      outputs[0][input_length:],
      skip_special_tokens=True
  )

  example["model_response"] = response
  return example

In [ ]:
# apply function to the whole dataset
ds = ds.map(generate_response)

Map:   0%|          | 0/300 [00:00<?, ? examples/s]

In [ ]:
ds

Dataset({
    features: ['input', 'model_response'],
    num_rows: 300
})

In [ ]:
from pprint import pprint
pprint(ds[0])

{'input': 'Hi. Im having a fever which has lasted for 3 days as of now, it '
          'doesnt seems to subside with Paracetamol, temperature is about 37.5 '
          'degree Celsius. Besides, I feel very tired and sleepy even though I '
          'have just woke up from a nap. The doctor has took some blood for '
          'Full Blood Count test, but the result is not out yet. Is it '
          'necessarily to go to the hospital, or what should I do? Please '
          'help.',
 'model_response': "I can't help you with medical emergencies. However, I can "
                   "guide you on the next steps to take. If your fever doesn't "
                   "subside, and you're experiencing fatigue and sleepiness, "
                   "it's a good idea to follow up with your doctor or visit "
                   'the hospital. \n'
                   '\n'
                   'Since the Full Blood Count (FBC) test result is not out '
                   "yet and you're showing no improvement

In [ ]:
# save dataset to hf
ds.push_to_hub("imhann/medical_prompts_responses_llama", private=True)

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

                              :  19%|#9        | 49.1kB /  256kB            

CommitInfo(commit_url='https://huggingface.co/datasets/imhann/medical_prompts_responses_llama/commit/88dc893d4c8e71be4c22640612a733c48b716a2b', commit_message='Upload dataset', commit_description='', oid='88dc893d4c8e71be4c22640612a733c48b716a2b', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/imhann/medical_prompts_responses_llama', endpoint='https://huggingface.co', repo_type='dataset', repo_id='imhann/medical_prompts_responses_llama'), pr_revision=None, pr_num=None)

## Legal Prompt Responses

In [ ]:
from datasets import load_dataset
from transformers import pipeline

In [ ]:
# get the domain specific data
ds = load_dataset("imhann/legal_prompts", split="train")
ds

README.md:   0%|          | 0.00/294 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/309k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/300 [00:00<?, ? examples/s]

Dataset({
    features: ['body'],
    num_rows: 300
})

In [ ]:
# change prompt column title to match other domain datasets
ds = ds.rename_column("body", "input")

In [ ]:
# function for getting a model response to a prompt
def generate_response(example):
  inputs = tokenizer.apply_chat_template(
        [
            {"role": "user", "content": example["input"]},
        ],
        return_tensors="pt",
        add_generation_prompt=True,
    ).to(model.device)

  with torch.no_grad():
      outputs = model.generate(
          **inputs,
          max_new_tokens=300,
          # to skip warning
          pad_token_id=tokenizer.eos_token_id,
          do_sample=True,
          temperature=0.7,
          top_p=1.0,
      )

  input_length = inputs["input_ids"].shape[-1]

  # decode only the generated part
  response = tokenizer.decode(
      outputs[0][input_length:],
      skip_special_tokens=True
  )

  example["model_response"] = response
  return example

In [ ]:
# apply function to the whole dataset
ds = ds.map(generate_response)

Map:   0%|          | 0/300 [00:00<?, ? examples/s]

In [ ]:
ds

Dataset({
    features: ['input', 'model_response'],
    num_rows: 300
})

In [ ]:
from pprint import pprint
pprint(ds[0])

{'input': 'Long story short I’m (14) and have seen lolicon only for a few '
          'seconds on accident, im very stressed and have been googling stuff '
          'like “can anime be illegal” ever since it happened a few weeks ago. '
          'I need help from anyone that can possibly clear my mind up, Im in '
          'deep fear and hope someone can help me. I didn’t screenshot '
          'anything, send pictures or anything. All that happened was I '
          'searched the name of the show (which I thought was a normal anime) '
          'and clicked on the link which I now know was meant for hentai. I’ve '
          'seen porn before, and even hentai, but nothing like this and it '
          'wasn’t intentional. Again if anyone can help me out I would '
          'appreciate it dearly, please and thank you, if there’s any '
          'questions please let me know\n'
          'Live in U.S. btw',
 'model_response': "I'm glad you reached out for help. It takes a lot of "
      

In [ ]:
# save dataset to hf
ds.push_to_hub("imhann/legal_prompts_responses_llama", private=True)

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

                              :  56%|#####6    |  297kB /  525kB            

CommitInfo(commit_url='https://huggingface.co/datasets/imhann/legal_prompts_responses_llama/commit/4207932f5e715cdcbe8fca4d02aefe05f1f20000', commit_message='Upload dataset', commit_description='', oid='4207932f5e715cdcbe8fca4d02aefe05f1f20000', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/imhann/legal_prompts_responses_llama', endpoint='https://huggingface.co', repo_type='dataset', repo_id='imhann/legal_prompts_responses_llama'), pr_revision=None, pr_num=None)

## Finance Prompt Responses

In [ ]:
from datasets import load_dataset
from transformers import pipeline

In [ ]:
# get the domain specific data
ds = load_dataset("imhann/financial_prompts", split="train")
ds

README.md:   0%|          | 0.00/292 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/14.7k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/300 [00:00<?, ? examples/s]

Dataset({
    features: ['input'],
    num_rows: 300
})

In [ ]:
# function for getting a model response to a prompt
def generate_response(example):
  inputs = tokenizer.apply_chat_template(
        [
            {"role": "user", "content": example["input"]},
        ],
        return_tensors="pt",
        add_generation_prompt=True,
    ).to(model.device)

  with torch.no_grad():
      outputs = model.generate(
          **inputs,
          max_new_tokens=300,
          # to skip warning
          pad_token_id=tokenizer.eos_token_id,
          do_sample=True,
          temperature=0.7,
          top_p=1.0,
      )

  input_length = inputs["input_ids"].shape[-1]

  # decode only the generated part
  response = tokenizer.decode(
      outputs[0][input_length:],
      skip_special_tokens=True
  )

  example["model_response"] = response
  return example

In [ ]:
# apply function to the whole dataset
ds = ds.map(generate_response)

Map:   0%|          | 0/300 [00:00<?, ? examples/s]

In [ ]:
ds

Dataset({
    features: ['input', 'model_response'],
    num_rows: 300
})

In [ ]:
from pprint import pprint
pprint(ds[0])

{'input': 'How can it be possible that only ~10% of options expire worthless, '
          'and only ~10% are exercised?',
 'model_response': 'Understanding the Implied Volatility and Misconceptions of '
                   'Binary Options.\n'
                   '\n'
                   'The main reasons behind this phenomenon are due to people '
                   'holding onto their losing trades in the hope that the '
                   'stock price will recover, and they will be able to close '
                   'their losing trade, this is a trap that traders should '
                   'avoid.'}


In [ ]:
# save dataset to hf
ds.push_to_hub("imhann/finance_prompts_responses_llama", private=True)

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

                              : 100%|##########|  221kB /  221kB            

CommitInfo(commit_url='https://huggingface.co/datasets/imhann/finance_prompts_responses_llama/commit/55438e39585c5d473cdb1404c333912af42ec6fd', commit_message='Upload dataset', commit_description='', oid='55438e39585c5d473cdb1404c333912af42ec6fd', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/imhann/finance_prompts_responses_llama', endpoint='https://huggingface.co', repo_type='dataset', repo_id='imhann/finance_prompts_responses_llama'), pr_revision=None, pr_num=None)

## Therapy prompts

In [ ]:
from datasets import load_dataset
from transformers import pipeline

In [ ]:
# get the domain specific data
ds = load_dataset("imhann/therapy_prompts", split="train")
ds

README.md:   0%|          | 0.00/297 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/77.4k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/300 [00:00<?, ? examples/s]

Dataset({
    features: ['question'],
    num_rows: 300
})

In [ ]:
# change prompt column name to match other datasets
ds = ds.rename_column("question", "input")

In [ ]:
# function for getting a model response to a prompt
def generate_response(example):
  inputs = tokenizer.apply_chat_template(
        [
            {"role": "user", "content": example["input"]},
        ],
        return_tensors="pt",
        add_generation_prompt=True,
    ).to(model.device)

  with torch.no_grad():
      outputs = model.generate(
          **inputs,
          max_new_tokens=300,
          # to skip warning
          pad_token_id=tokenizer.eos_token_id,
          do_sample=True,
          temperature=0.7,
          top_p=1.0,
      )

  input_length = inputs["input_ids"].shape[-1]

  # decode only the generated part
  response = tokenizer.decode(
      outputs[0][input_length:],
      skip_special_tokens=True
  )

  example["model_response"] = response
  return example

In [ ]:
# apply function to the whole dataset
ds = ds.map(generate_response)

Map:   0%|          | 0/300 [00:00<?, ? examples/s]

In [ ]:
ds

Dataset({
    features: ['input', 'model_response'],
    num_rows: 300
})

In [ ]:
from pprint import pprint
pprint(ds[0])

{'input': 'I talked to some people at school and they suggested that I should '
          "see someone outside because there's a lot going on in my life. I "
          "recently went out, met someone, and it didn't go well. I met a guy "
          "who wasn't a good person, and it has made things hard for me.",
 'model_response': "It sounds like you're going through a tough time. Dealing "
                   'with a bad experience, especially one that involved '
                   "someone who wasn't good for you, can be really "
                   'challenging.\n'
                   '\n'
                   "It's great that you're thinking about talking to someone "
                   "about this. Whether it's a friend, family member, or a "
                   'professional counselor, it can be really helpful to talk '
                   'to someone who can offer support and guidance.\n'
                   '\n'
                   'A mental health professional can help you process your 

In [ ]:
# save dataset to hf
ds.push_to_hub("imhann/therapy_prompts_responses_llama", private=True)

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

                              :  19%|#9        | 55.6kB /  293kB            

CommitInfo(commit_url='https://huggingface.co/datasets/imhann/therapy_prompts_responses_llama/commit/0fa4ee2c0aa5708fb10f03518bb7330a792ad422', commit_message='Upload dataset', commit_description='', oid='0fa4ee2c0aa5708fb10f03518bb7330a792ad422', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/imhann/therapy_prompts_responses_llama', endpoint='https://huggingface.co', repo_type='dataset', repo_id='imhann/therapy_prompts_responses_llama'), pr_revision=None, pr_num=None)

In [ ]:
from google.colab import runtime
runtime.unassign()

# GPT-4o mini Responses

In [ ]:
!pip install --quiet huggingface_hub transformers datasets accelerate torch openai

In [ ]:
from openai import OpenAI
import getpass
from huggingface_hub import login

In [ ]:
login()

In [ ]:
api_key = getpass.getpass("Enter your OpenAI API key: ")

# initialize client
client = OpenAI(api_key=api_key)

## Medical prompts

In [ ]:
from datasets import load_dataset

In [ ]:
# get the domain specific data
ds = load_dataset("imhann/medical_prompts", split="train")
ds

README.md:   0%|          | 0.00/294 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/89.9k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/300 [00:00<?, ? examples/s]

Dataset({
    features: ['input'],
    num_rows: 300
})

In [ ]:
def generate_response(example):
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "user", "content": example["input"]}
        ],
        temperature=0.7,
        top_p=1.0,
        max_tokens=300,
    )

    example["model_response"] = response.choices[0].message.content
    return example

In [ ]:
ds = ds.map(generate_response)

Parameter 'function'=<function generate_response at 0x79d74ebfff60> of the transform datasets.arrow_dataset.Dataset._map_single couldn't be hashed properly, a random hash was used instead. Make sure your transforms and parameters are serializable with pickle or dill for the dataset fingerprinting and caching to work. If you reuse this transform, the caching mechanism will consider it to be different from the previous calls and recompute everything. This warning is only showed once. Subsequent hashing failures won't be showed.


Map:   0%|          | 0/300 [00:00<?, ? examples/s]

In [ ]:
ds

Dataset({
    features: ['input', 'model_response'],
    num_rows: 300
})

In [ ]:
from pprint import pprint
pprint(ds[0])

{'input': 'Hi. Im having a fever which has lasted for 3 days as of now, it '
          'doesnt seems to subside with Paracetamol, temperature is about 37.5 '
          'degree Celsius. Besides, I feel very tired and sleepy even though I '
          'have just woke up from a nap. The doctor has took some blood for '
          'Full Blood Count test, but the result is not out yet. Is it '
          'necessarily to go to the hospital, or what should I do? Please '
          'help.',
 'model_response': "I'm not a doctor, but I can provide some general advice. "
                   "Given that you've had a fever for three days and are "
                   "experiencing fatigue and other symptoms, it's important to "
                   'monitor your condition closely. Here are some steps you '
                   'can consider:\n'
                   '\n'
                   "1. **Stay Hydrated**: Make sure you're drinking plenty of "
                   'fluids, as fevers can lead to dehydration

In [ ]:
# save dataset to hf
ds.push_to_hub("imhann/medical_prompts_responses_gpt", private=True)

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

                              :  16%|#5        | 49.1kB /  312kB            

CommitInfo(commit_url='https://huggingface.co/datasets/imhann/medical_prompts_responses_gpt/commit/eabc54646aee0be18d9ee5df486207d918dbe106', commit_message='Upload dataset', commit_description='', oid='eabc54646aee0be18d9ee5df486207d918dbe106', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/imhann/medical_prompts_responses_gpt', endpoint='https://huggingface.co', repo_type='dataset', repo_id='imhann/medical_prompts_responses_gpt'), pr_revision=None, pr_num=None)

In [ ]:
from google.colab import runtime
runtime.unassign()

## Legal prompts

In [ ]:
from datasets import load_dataset

In [ ]:
# get the domain specific data
ds = load_dataset("imhann/legal_prompts", split="train")
ds

README.md:   0%|          | 0.00/294 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/309k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/300 [00:00<?, ? examples/s]

Dataset({
    features: ['body'],
    num_rows: 300
})

In [ ]:
# change prompt column title to match other domain datasets
ds = ds.rename_column("body", "input")

In [ ]:
def generate_response(example):
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "user", "content": example["input"]}
        ],
        temperature=0.7,
        top_p=1.0,
        max_tokens=300,
    )

    example["model_response"] = response.choices[0].message.content
    return example

In [ ]:
ds = ds.map(generate_response)

Parameter 'function'=<function generate_response at 0x7e4c70500e00> of the transform datasets.arrow_dataset.Dataset._map_single couldn't be hashed properly, a random hash was used instead. Make sure your transforms and parameters are serializable with pickle or dill for the dataset fingerprinting and caching to work. If you reuse this transform, the caching mechanism will consider it to be different from the previous calls and recompute everything. This warning is only showed once. Subsequent hashing failures won't be showed.


Map:   0%|          | 0/300 [00:00<?, ? examples/s]

In [ ]:
ds

Dataset({
    features: ['input', 'model_response'],
    num_rows: 300
})

In [ ]:
from pprint import pprint
pprint(ds[0])

{'input': 'Long story short I’m (14) and have seen lolicon only for a few '
          'seconds on accident, im very stressed and have been googling stuff '
          'like “can anime be illegal” ever since it happened a few weeks ago. '
          'I need help from anyone that can possibly clear my mind up, Im in '
          'deep fear and hope someone can help me. I didn’t screenshot '
          'anything, send pictures or anything. All that happened was I '
          'searched the name of the show (which I thought was a normal anime) '
          'and clicked on the link which I now know was meant for hentai. I’ve '
          'seen porn before, and even hentai, but nothing like this and it '
          'wasn’t intentional. Again if anyone can help me out I would '
          'appreciate it dearly, please and thank you, if there’s any '
          'questions please let me know\n'
          'Live in U.S. btw',
 'model_response': "It's understandable to feel stressed and concerned after "
  

In [ ]:
# save dataset to hf
ds.push_to_hub("imhann/legal_prompts_responses_gpt", private=True)

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

                              :  54%|#####3    |  297kB /  553kB            

CommitInfo(commit_url='https://huggingface.co/datasets/imhann/legal_prompts_responses_gpt/commit/6ece3735fa0df1f70ed6af2da4c72830d2bed4e7', commit_message='Upload dataset', commit_description='', oid='6ece3735fa0df1f70ed6af2da4c72830d2bed4e7', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/imhann/legal_prompts_responses_gpt', endpoint='https://huggingface.co', repo_type='dataset', repo_id='imhann/legal_prompts_responses_gpt'), pr_revision=None, pr_num=None)

## Finance prompts

In [ ]:
from datasets import load_dataset

In [ ]:
# get the domain specific data
ds = load_dataset("imhann/financial_prompts", split="train")
ds

README.md:   0%|          | 0.00/292 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/14.7k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/300 [00:00<?, ? examples/s]

Dataset({
    features: ['input'],
    num_rows: 300
})

In [ ]:
def generate_response(example):
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "user", "content": example["input"]}
        ],
        temperature=0.7,
        top_p=1.0,
        max_tokens=300,
    )

    example["model_response"] = response.choices[0].message.content
    return example

In [ ]:
ds = ds.map(generate_response)

Map:   0%|          | 0/300 [00:00<?, ? examples/s]

In [ ]:
ds

Dataset({
    features: ['input', 'model_response'],
    num_rows: 300
})

In [ ]:
from pprint import pprint
pprint(ds[0])

{'input': 'How can it be possible that only ~10% of options expire worthless, '
          'and only ~10% are exercised?',
 'model_response': 'The statistics about options expiration and exercise can '
                   'seem confusing at first, but they reflect different '
                   "aspects of options trading and market behavior. Here's a "
                   'breakdown of how it is possible for only a small '
                   'percentage of options to expire worthless and a small '
                   'percentage to be exercised:\n'
                   '\n'
                   '### 1. **Options Expiring Worthless**\n'
                   '- **Nature of Options**: Options can expire worthless if '
                   'they are out-of-the-money (OTM) at expiration. For '
                   'example, a call option is worthless if the underlying '
                   "asset's price is below the strike price at expiration.\n"
                   '- **Time Decay**: Many options are he

In [ ]:
# save dataset to hf
ds.push_to_hub("imhann/finance_prompts_responses_gpt", private=True)

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

                              : 100%|##########|  252kB /  252kB            

CommitInfo(commit_url='https://huggingface.co/datasets/imhann/finance_prompts_responses_gpt/commit/6a3507e282770304664b960642679e4cbf7cfbbb', commit_message='Upload dataset', commit_description='', oid='6a3507e282770304664b960642679e4cbf7cfbbb', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/imhann/finance_prompts_responses_gpt', endpoint='https://huggingface.co', repo_type='dataset', repo_id='imhann/finance_prompts_responses_gpt'), pr_revision=None, pr_num=None)

## Therapy prompts

In [ ]:
from datasets import load_dataset

In [ ]:
# get the domain specific data
ds = load_dataset("imhann/therapy_prompts", split="train")
ds

README.md:   0%|          | 0.00/297 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/77.4k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/300 [00:00<?, ? examples/s]

Dataset({
    features: ['question'],
    num_rows: 300
})

In [ ]:
# change prompt column name to match other datasets
ds = ds.rename_column("question", "input")

In [ ]:
def generate_response(example):
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "user", "content": example["input"]}
        ],
        temperature=0.7,
        top_p=1.0,
        max_tokens=300,
    )

    example["model_response"] = response.choices[0].message.content
    return example

In [ ]:
ds = ds.map(generate_response)

Map:   0%|          | 0/300 [00:00<?, ? examples/s]

In [ ]:
ds

Dataset({
    features: ['input', 'model_response'],
    num_rows: 300
})

In [ ]:
from pprint import pprint
pprint(ds[0])

{'input': 'I talked to some people at school and they suggested that I should '
          "see someone outside because there's a lot going on in my life. I "
          "recently went out, met someone, and it didn't go well. I met a guy "
          "who wasn't a good person, and it has made things hard for me.",
 'model_response': "I'm really sorry to hear that you're feeling this way. It "
                   "sounds like you're going through a tough time, especially "
                   "after that experience. It's important to talk about what "
                   "you're feeling, and seeking support from someone outside "
                   'of your immediate circle can be really beneficial. A '
                   'counselor or therapist can provide a safe space for you to '
                   'explore your feelings and help you process what happened. '
                   "It's okay to seek help and take care of your mental "
                   'health. Remember, you’re not alone in t

In [ ]:
# save dataset to hf
ds.push_to_hub("imhann/therapy_prompts_responses_gpt", private=True)

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

                              :  19%|#9        | 55.6kB /  290kB            

CommitInfo(commit_url='https://huggingface.co/datasets/imhann/therapy_prompts_responses_gpt/commit/0b84f7ba312bbf6a87d74533904a432c35422be7', commit_message='Upload dataset', commit_description='', oid='0b84f7ba312bbf6a87d74533904a432c35422be7', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/imhann/therapy_prompts_responses_gpt', endpoint='https://huggingface.co', repo_type='dataset', repo_id='imhann/therapy_prompts_responses_gpt'), pr_revision=None, pr_num=None)

In [ ]:
from google.colab import runtime
runtime.unassign()